# pandas 之四：DataFrame 入门

---

## DataFrame 是什么

**一句话：`DataFrame` = 带行标签和列标签的二维表格。**

就是你在 Excel 里最熟悉的那种样子：

```
        数学  英语  语文     <- 列索引（columns）
  小明    90   88   78
  小红    85   92   80       <- 行索引（index）
  小刚    95   80   85
```

和 Excel 的区别在于：**DataFrame 的每一列可以是不同的数据类型**
（一列整数、一列文字、一列日期，同时存在没问题），
而且能用代码批量操作。

## 和 Series 的关系

| | 维度 | 类比 |
|---|---|---|
| `Series` | 1 维 | **一列** |
| `DataFrame` | 2 维 | **一张表** |

**一个 DataFrame 里的每一列，单独取出来就是一个 Series。**

## 学习目标

1. 掌握 3 种创建方式：字典、Series、嵌套结构
2. 弄清 `index` / `columns` / `dtypes` 等核心属性
3. 掌握取行、取列、取单元格、条件筛选
4. 知道 `loc` 和 `iloc` 在二维里怎么用
5. 会做排序、统计、增删改


In [ ]:
import pandas as pd
import numpy as np

print('pandas 版本：', pd.__version__)


---
## 一、创建 DataFrame

### 方式 1：从字典创建（最常用）

**规则：字典的"键"变成列名，"值"（列表）变成该列的数据。**


In [ ]:
df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'age': [23, 30, 25, 35, 28],
    'gender': ['F', 'M', 'M', 'M', 'F'],
})
print(df)


### ⚠️ 关键约束：每列的数据长度必须一致

因为字典里每一列是一个"列表"，这些列表**长度必须相等**，
否则无法拼成矩形表格。


In [ ]:
try:
    pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5]})
except ValueError as e:
    print('列长度不一致 -> 报错：')
    print(e)


### 方式 2：从多个 Series 创建（会自动"按索引对齐"）

这是理解 pandas 的关键机制：**不是简单拼接，而是按索引对齐**。


In [ ]:
s1 = pd.Series(['a', 'b', 'c'], index=[1, 2, 3])
s2 = pd.Series([10, 20, 30], index=[1, 2, 3])

df2 = pd.DataFrame({'第一列': s1, '第二列': s2})
print(df2)


### ⚠️ 索引不一致 -> 自动出现 NaN

如果两个 Series 的索引不一样，pandas **不会报错**，
而是取"索引的并集"，**对不上的位置填 `NaN`**。

这是 DataFrame 里出现 NaN 的**头号原因**，必须理解。


In [ ]:
s3 = pd.Series([1, 2, 3, 4, 5], index=['a', 'b', 'c', 'd', 'e'])
s4 = pd.Series([6, 7, 8, 9, 10], index=['f', 'g', 'h', 'i', 'j'])

# 两个 Series 的索引完全没有交集，还额外指定了行索引
df3 = pd.DataFrame({'A': s3, 'B': s4}, index=[f'row_{i}' for i in range(1, 6)])
print('索引对不上 -> 全是 NaN：')
print(df3)


**图解**：

```
指定的行索引:  row_1  row_2  ...      <- 想用的行标签
s3 的索引:     a      b     ...       <- 两者毫无交集
结果:          全部 NaN
```

**结论**：`index` 参数会"覆盖"原始索引，
原有的值找不到对应位置，就全部变成 `NaN`。


In [ ]:
# 正确做法：不让索引冲突（要么不指定 index，要么让 index 与原始索引一致）
df4 = pd.DataFrame({'A': s3, 'B': s4})
print('不指定 index 时：pandas 取两个索引的并集')
print(df4)


### 方式 3：指定 `index` 和 `columns`


In [ ]:
data = [[112, 113, 115],
        [116, 117, 118],
        [120, 121, 122]]

df5 = pd.DataFrame(
    data,
    index=['张三', '李四', '王五'],
    columns=['math', 'english', 'chinese'],
)
print(df5)


### 方式 4：从 Numpy 数组创建


In [ ]:
arr = np.arange(1, 13).reshape(3, 4)
df_arr = pd.DataFrame(arr, columns=['A', 'B', 'C', 'D'])
print(df_arr)
print('\n注意索引是默认的 0,1,2 —— pandas 不会替你生成"有意义的"索引')


---
## 二、核心属性

| 属性 | 含义 |
|---|---|
| `df.shape` | (行数, 列数) |
| `df.index` | 行索引 |
| `df.columns` | 列名 |
| `df.dtypes` | 每列的数据类型 |
| `df.size` | 元素总个数 = 行 × 列 |
| `df.ndim` | 维度（永远是 2） |
| `df.values` | 底层 Numpy 数组 |
| `df.T` | 转置（行列互换） |


In [ ]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9],
})

print('shape   :', df.shape, '  <- 3 行 3 列')
print('index   :', list(df.index))
print('columns :', list(df.columns))
print('size    :', df.size, '  <- 3×3=9')
print('ndim    :', df.ndim)
print('\ndtypes（注意这是"每列一个类型"）：')
print(df.dtypes)
print('\nvalues：')
print(df.values)
print('类型：', type(df.values))


In [ ]:
print('T 转置（行变列、列变行）：')
print(df.T)


### `df.info()` 和 `df.describe()`：上手先看这两个

拿到一份陌生数据，**第一件事就是跑这两个方法**。


In [ ]:
# 造一份"有缺失值、有不同类型"的数据，更贴近真实情况
data = {
    'Name': ['Alice', 'Bob', None, 'David'],
    'Age': [25, 25, 30, 22],
    'City': ['New York', 'Los Angeles', 'Chicago', None],
}
df_real = pd.DataFrame(data)
print(df_real)


In [ ]:
print('=== df.info() ===')
print('看：有几行、每列叫什么、多少非空值、什么类型、占多少内存')
df_real.info()


**`Non-Null Count` 是关键**：这里是 `3 non-null`，而总共 `4 entries`，
说明**有 1 个缺失值**。这是发现缺失值最快的方式。


In [ ]:
print('=== df.describe() ===')
print('默认只统计【数值列】')
print(df_real.describe())


如果想连文字列一起看，加 `include='all'`：


In [ ]:
print(df_real.describe(include='all'))


### 检查缺失值：`isnull()` / `isna()`


In [ ]:
print('df_real.isna()：')
print(df_real.isna())

print('\n按【列】统计缺失值个数（axis=0，默认）：')
print(df_real.isna().sum(axis=0))

print('\n按【行】统计缺失值个数（axis=1）：')
print(df_real.isna().sum(axis=1))


---
## 三、取值：取列、取行、取单元格

这是 DataFrame 里**最需要练熟**的部分。先给一个清晰的表：

| 想取什么 | 写法 |
|---|---|
| **一列** | `df['列名']` -> 返回 **Series** |
| **多列** | `df[['列1','列2']]` -> 返回 **DataFrame** |
| **一行（按标签）** | `df.loc['行标签']` |
| **一行（按位置）** | `df.iloc[0]` |
| **一个单元格** | `df.loc['行','列']` 或 `df.at['行','列']` |
| **一块区域** | `df.loc['行范围','列范围']` |


In [ ]:
print('取一列 df["A"]（Series）：')
print(df['A'])
print('类型：', type(df['A']))

print('\n取多列 df[["A","B"]]（DataFrame）：')
print(df[['A', 'B']])
print('类型：', type(df[['A', 'B']]))


> **注意方括号的层数！**
> - `df['A']` 一层 -> **Series**
> - `df[['A']]` 两层 -> **DataFrame**
>
> 少一层多一层，返回类型完全不同，后面的操作能不能用也随之改变。
> **这个区别在真实代码里非常关键。**


### 取行：`loc`（标签）和 `iloc`（位置）


In [ ]:
named = pd.DataFrame([[1, 4, 7], [2, 5, 8], [3, 6, 9]],
                     index=['r1', 'r2', 'r3'],
                     columns=['A', 'B', 'C'])
print(named)

print("\nloc['r1'] 按标签取行：")
print(named.loc['r1'])

print('\niloc[0] 按位置取行：')
print(named.iloc[0])


### 取单元格：`at` / `iat`（比 loc/iloc 快）


In [ ]:
print("named.at['r2','B']  =", named.at['r2', 'B'])
print("named.iat[1, 1]     =", named.iat[1, 1])
print('两者取的是同一个格子（r2 行 B 列，值 5）')


### 取一块区域


In [ ]:
print("named.loc['r1':'r2', 'A':'B']  <- loc 切片【包含】两端：")
print(named.loc['r1':'r2', 'A':'B'])

print("\nnamed.iloc[0:2, 0:2]  <- iloc 切片【不含】末尾：")
print(named.iloc[0:2, 0:2])


---
## 四、条件筛选

**套路：先造一个布尔 Series，再塞进 `df[...]`。**


In [ ]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9],
})

# 单条件
print('A > 1 的行：')
print(df[df['A'] > 1])


### ⚠️ 多条件：必须用 `&` `|`，并且每个条件加括号


In [ ]:
print('A > 1 且 B < 6：')
print(df[(df['A'] > 1) & (df['B'] < 6)])

print('\nA > 2 或 B < 5：')
print(df[(df['A'] > 2) | (df['B'] < 5)])


**为什么不能用 `and` / `or`？**
因为它们只能处理单个布尔值，而这里每个条件是**一整个布尔数组**。
用 `and` 会直接抛错。**这是 pandas 新手最常犯的错。**


In [ ]:
try:
    df[(df['A'] > 1) and (df['B'] < 6)]
except ValueError as e:
    print('用 and -> 报错：')
    print(str(e)[:200])


### 用 `isin()` 筛选"值在某个列表里"


In [ ]:
print("A 在 [1, 3] 里的行：")
print(df[df['A'].isin([1, 3])])


### `isin` 也支持传入一个集合，效率略高

> **原稿的小问题**：原稿写的是 `print(df.isin({"Alice", 25}))`。
> 这个调用**语法上没错**，但它检查的是"整张表里每个格子是否等于 Alice 或 25"，
> 返回的是一张**全是 True/False 的表**，而不是"筛出符合条件的行"。
> 想筛选行，得用 `df[df['某列'].isin([...])]` 的写法。
> 下面把这两种用法都演示一遍，对比就清楚了。


In [ ]:
df_people = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 25],
})
print('df.isin(...) 返回的是"同形状的布尔表"：')
print(df_people.isin({'Alice', 25}))

print('\n真正的"筛选行"要这样写：')
print(df_people[df_people['age'] == 25])


---
## 五、增删改


In [ ]:
d = pd.DataFrame([[112, 113], [116, 117]], index=['张三', '李四'],
                 columns=['math', 'english'])
print(d)


### 新增列


In [ ]:
# 方式 1：直接赋值（在最后加一列）
d['pe'] = [200, 201]
print('直接赋值新增列：')
print(d)

# 方式 2：用现有列计算生成新列
d['total'] = d['math'] + d['english']
print('\n用已有列算出新列：')
print(d)


In [ ]:
# 方式 3：insert 插到指定位置
d.insert(1, 'physics', [300, 301])
print('insert(1, ...) 插到第 1 列位置：')
print(d)


### 新增行


In [ ]:
# 注意：此时 d 已经有 5 列了（前面 insert 过 physics）
print('当前列：', list(d.columns))

# 用字典新增一行最稳：不用关心列的顺序，也不怕列数变化
d.loc['王五'] = {'math': 123, 'physics': 130, 'english': 124, 'pe': 125, 'total': 126}
print('\n用 loc + 字典新增一行：')
print(d)


> **为什么推荐用字典而不是列表？**
> 用列表 `d.loc['王五'] = [123, 124, ...]` 时，你必须**牢记列的当前顺序和数量**。
> 一旦前面加过列（比如刚才 `insert` 了 `physics`），数量对不上就会报
> `ValueError: cannot set a row with mismatched columns`。
>
> **用字典则按列名赋值，顺序无关、少写一列也不会错。** 这是更抗改动的写法。


### 合并多行：`pd.concat`


In [ ]:
d2 = pd.DataFrame({
    'math': [127], 'physics': [130], 'english': [128], 'pe': [127], 'total': [255],
}, index=['赵六'])
print('要追加的行：')
print(d2)

print('\nconcat 拼接后：')
print(pd.concat([d, d2]))


### 修改列名 / 行名


In [ ]:
d3 = pd.DataFrame([[1, 2], [3, 4]], columns=['a', 'b'], index=['x', 'y'])

# 方式 1：整体替换（必须列数对上）
d3.columns = ['数学', '英语']
print('整体改列名：')
print(d3)


In [ ]:
# 方式 2：rename 局部改（推荐，安全）
d4 = pd.DataFrame([[1, 2], [3, 4]], columns=['a', 'b'], index=['x', 'y'])
print('rename 局部改：')
print(d4.rename(columns={'a': '甲'}, index={'x': '第一行'}))


> **`rename` 比"整体赋值 columns" 安全**，因为：
> - 只改你想改的，其余不动
> - 列数对不上也不会出错（整体赋值要求长度完全一致）
>
> 另外 `rename` 默认**返回新对象**，想改原件要加 `inplace=True`（或者接收返回值）。


### 删除行 / 列：`drop`


In [ ]:
d5 = pd.DataFrame([[1, 2, 3], [4, 5, 6], [7, 8, 9]],
                  index=['r1', 'r2', 'r3'], columns=['A', 'B', 'C'])
print('原始：')
print(d5)

print('\n删列（columns=）：')
print(d5.drop(columns=['B']))

print('\n删行（index=）：')
print(d5.drop(index='r2'))


> **`drop` 默认返回新对象，原表不变**。
> 想直接改原表要写 `inplace=True`。
> **练手的代码里建议不要用 inplace** —— 一旦删错还能回头。


In [ ]:
print('d5 还在吗？')
print(d5, '  <- 原表没变')


### 删除"满足条件的行"


In [ ]:
d6 = pd.DataFrame({'姓名': ['张三', '李四', '王五'], '成绩': [85, 52, 90]})
print('原始：')
print(d6)

# 思路：先找出满足条件的行的索引，再 drop
bad_idx = d6[d6['成绩'] < 60].index
print('\n不及格的索引：', list(bad_idx))
print('删掉之后：')
print(d6.drop(index=bad_idx))


### 累加与累计最大


In [ ]:
d7 = pd.DataFrame({'A': [1, 2, 3], 'B': [10, 20, 30]})
print('cumsum 累计和：')
print(d7.cumsum())

print('\ncummax 累计最大（一路记录历史最高）：')
print(d7.cummax())


> **原稿的问题**：原稿的 `cumsum` / `cummax` / `sum` 例子是在一个
> **同时含字符串和数字的 DataFrame** 上跑的，
> 在 pandas 3.0 里这种混合类型求和会**直接报错**或得到无意义的结果。
> 这里改成了纯数值表，语义才清楚。


---
## 六、排序


In [ ]:
d8 = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '年龄': [28, 35, 23],
    '分数': [85, 92, 78],
}, index=['c', 'a', 'b'])

print('按【行索引】排序：')
print(d8.sort_index())

print('\n按【列名】排序（axis=1）：')
print(d8.sort_index(axis=1))

print('\n按【分数】列的值排序（降序）：')
print(d8.sort_values(by='分数', ascending=False))


In [ ]:
# nlargest / nsmallest：直接取前 N
print('分数最高的 2 人：')
print(d8.nlargest(2, '分数'))

print('\n分数最低的 2 人：')
print(d8.nsmallest(2, '分数'))


---
## 七、综合案例

### 案例 1：学生成绩


In [ ]:
score = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六', '钱七'],
    '数学': [95, 88, 99, 76, 85],
    '英语': [78, 92, 85, 89, 90],
    '物理': [88, 79, 94, 91, 87],
})
print(score)


**1) 计算每位学生的总分和平均分**


In [ ]:
subjects = ['数学', '英语', '物理']
score['总分'] = score[subjects].sum(axis=1)      # axis=1 -> 按行求和
score['平均分'] = score[subjects].mean(axis=1)

print(score[['姓名', '总分', '平均分']])


**2) 数学 > 98 或 英语 > 85 的学生**


In [ ]:
filtered = score[(score['数学'] > 98) | (score['英语'] > 85)]
print(filtered[['姓名', '数学', '英语']])


**3) 按总分降序，取前三名**

两种写法都对，**推荐 `nlargest`**（语义更直白）：


In [ ]:
print('写法 A：nlargest')
print(score.nlargest(3, '总分')[['姓名', '总分', '平均分']])

print('\n写法 B：sort_values + head')
print(score.sort_values(by='总分', ascending=False).head(3)[['姓名', '总分', '平均分']])


### 案例 2：产品销售


In [ ]:
sales = pd.DataFrame({
    '产品': ['手机', '电脑', '平板', '耳机', '智能手表'],
    '单价': [3000, 5000, 2000, 500, 1500],
    '销量': [150, 80, 120, 300, 90],
})

sales['销售额'] = sales['单价'] * sales['销量']
print('1) 各产品销售额：')
print(sales[['产品', '销售额']])

print('\n2) 销售额最高的两种产品：')
print(sales.nlargest(2, '销售额')[['产品', '销售额']])

print('\n3) 按销售额从高到低排序：')
print(sales.sort_values(by='销售额', ascending=False))


### 案例 3：电商用户消费


In [ ]:
consum = pd.DataFrame({
    '用户ID': [101, 102, 103, 104, 105],
    '用户名': ['张三', '李四', '王五', '赵六', '钱七'],
    '消费金额': [250.75, 300.50, 150.00, 400.25, 350.80],
    '商品类别': ['电子产品', '服装', '电子产品', '家居', '电子产品'],
    '商品单价': [500.00, 150.00, 300.00, 200.00, 700.00],
    '购买数量': [1, 2, 1, 2, 1],
})

consum['消费总金额'] = consum['消费金额'] * consum['购买数量']
print('1) 每位用户的总消费：')
print(consum[['用户ID', '用户名', '消费总金额']])

print('\n2) 消费最高的用户（全部信息）：')
print(consum.nlargest(1, '消费总金额').T, '  <- 用 .T 转置更好看')

print('\n3) 平均消费金额：', round(consum['消费总金额'].mean(), 2))

print('\n4) 电子产品的购买数量合计：')
elec = consum[consum['商品类别'] == '电子产品']
print(elec['购买数量'].sum())


---
## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 多条件用 `and` / `or` | `ValueError` | 用 `&` `|`，且每个条件加括号 |
| `df['A']` 和 `df[['A']]` 分不清 | 返回类型不同 | 一层是 Series，两层是 DataFrame |
| 用 Series 建表出现满屏 NaN | 索引对不上 | 索引对齐机制；检查两边 `index` |
| 字典建表列长度不等 | `ValueError` | 每列列表长度必须一致 |
| `df.isin([...])` 以为能筛行 | 返回布尔表 | 筛行要 `df[df[列].isin([...])]` |
| `drop` / `rename` 以为改了原表 | 原表没变 | 默认返回新对象；要改原件加 `inplace=True` |
| `axis` 记反 | 结果方向不对 | `axis=0` 竖着算（按列），`axis=1` 横着算（按行） |
| 混类型列做 `sum` / `cumsum` | 报错或结果无意义 | 只用数值列参与计算 |
| `loc` 切片以为不含末尾 | 多取了一个 | `loc` **两端都包含** |
| 拿到新数据先 `head()` 看 | 盲猜结构 | 先跑 `info()` 和 `describe()` |

---

## 九、动手练习

**练习 1**：用字典创建一个 4 行 3 列的 DataFrame（列名 `姓名/年龄/城市`），行索引设为 `['a','b','c','d']`，打印 `shape`、`columns`、`dtypes`。

**练习 2**：给定上表，分别取出：①`姓名` 列；②`姓名`+`城市` 两列；③`b` 行；④`b` 行 `城市` 列那个单元格。

**练习 3**：筛选出"年龄 > 25 且城市是北京"的行（先自己想好括号怎么加）。

**练习 4**：给表格新增一列 `年龄分组`，用 `pd.cut` 把年龄分成 `青年/中年/老年`。

**练习 5**：把 `年龄` 列改名为 `age`，然后按 `age` 降序排序。

**练习 6**：删除"年龄 < 20"的所有行（提示：先用条件拿到 `.index`，再 `drop`）。

**练习 7**：某班级 5 人成绩如下，求每人总分、平均分，并找出"总分最高的 2 人"：
数学 `[88,92,79,95,85]`、英语 `[76,88,90,82,91]`。

---

**答案思路（先自己做）**

```python
# 1
df = pd.DataFrame({'姓名': ['张三','李四','王五','赵六'],
                   '年龄': [28, 35, 23, 42],
                   '城市': ['北京','上海','北京','广州']},
                  index=['a','b','c','d'])
print(df.shape, list(df.columns), df.dtypes.to_dict())

# 2
print(df['姓名'])                  # Series
print(df[['姓名','城市']])          # DataFrame
print(df.loc['b'])
print(df.loc['b', '城市'])

# 3
print(df[(df['年龄'] > 25) & (df['城市'] == '北京')])

# 4
df['年龄分组'] = pd.cut(df['年龄'], bins=[0, 30, 50, 100],
                        labels=['青年','中年','老年'])

# 5
print(df.rename(columns={'年龄': 'age'}).sort_values(by='age', ascending=False))

# 6
print(df.drop(index=df[df['年龄'] < 20].index))

# 7
s = pd.DataFrame({'姓名': ['甲','乙','丙','丁','戊'],
                  '数学': [88,92,79,95,85],
                  '英语': [76,88,90,82,91]})
s['总分'] = s[['数学','英语']].sum(axis=1)
s['平均分'] = s[['数学','英语']].mean(axis=1)
print(s.nlargest(2, '总分'))
```
